# NOT on Colored MNIST 2 → 3 (Multi-colored)

Every source '2' and target '3' gets a random hue. The transport map is deterministic and the images are 16×16.

**How to run.** Run the setup cells (*Data*, *Architecture*, *Training with `src.train`*), then exactly one training section: it seeds everything with `SEED` and builds a fresh `T` and `f` with `build_models()`, so the sections are independent. Finish with *Evaluation*, which scores the model trained last. Each section reads its step sizes from `LR` in the *Architecture* cell; the TTS-EG sections differ from EG only in the step-size ratio ($\gamma_T > \gamma_V$ for Semi-Dual, $\gamma_T < \gamma_V$ for Lagrangian).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as datasets
from torchvision import transforms

from IPython.display import clear_output
from tqdm.notebook import tqdm as tqdm
import ot
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Data

In [ ]:
BATCH_SIZE = 16

def random_color(im):
    hue = 360*np.random.rand()
    d = (im *(hue%60)/60)
    im_min, im_inc, im_dec = torch.zeros_like(im), d, im - d
    c_im = torch.zeros((3, im.shape[1], im.shape[2]))
    H = round(hue/60) % 6
    cmap = [[0, 3, 2], [2, 0, 3], [1, 0, 3], [1, 2, 0], [3, 1, 0], [0, 1, 2]]
    return torch.cat((im, im_min, im_dec, im_inc), dim=0)[cmap[H]]

TRANSFORM = transforms.Compose([
    transforms.Resize(16),
    transforms.ToTensor(),
    random_color,
    transforms.Normalize([0.5],[0.5])
])

# Load train datasets
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=TRANSFORM)
idx = np.array(range(len(mnist_train)))
mnist_2 = torch.utils.data.Subset(mnist_train, idx[mnist_train.targets==2])
mnist_3 = torch.utils.data.Subset(mnist_train, idx[mnist_train.targets==3])
mnist_2_loader = torch.utils.data.DataLoader(mnist_2, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
mnist_3_loader = torch.utils.data.DataLoader(mnist_3, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# Load test datasets
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=TRANSFORM)
idx = np.array(range(len(mnist_test)))
mnist_2_test = torch.utils.data.Subset(mnist_test, idx[mnist_test.targets==2])
mnist_3_test = torch.utils.data.Subset(mnist_test, idx[mnist_test.targets==3])
mnist_2_test_loader = torch.utils.data.DataLoader(mnist_2_test, batch_size=BATCH_SIZE)
mnist_3_test_loader = torch.utils.data.DataLoader(mnist_3_test, batch_size=BATCH_SIZE)

# We pick a few samples from them for the qualitative analysis
X_test_fixed = next(iter(mnist_2_test_loader))[0]
Y_test_fixed = next(iter(mnist_3_test_loader))[0]


In [ ]:
iter_mnist_2, iter_mnist_3 = iter(mnist_2_loader), iter(mnist_3_loader)

def sample_mnist_2():
    global iter_mnist_2, mnist_2_loader
    try:
        return next(iter_mnist_2)[0]
    except StopIteration:
        iter_mnist_2 = iter(mnist_2_loader)
        return next(iter_mnist_2)[0]

def sample_mnist_3():
    global iter_mnist_3, mnist_3_loader
    try:
        return next(iter_mnist_3)[0]
    except StopIteration:
        iter_mnist_3 = iter(mnist_3_loader)
        return next(iter_mnist_3)[0]

In [ ]:
def plot_images(batch):
    batch = batch.cpu()
    fig, axes = plt.subplots(1, 10, figsize=(10, 1), dpi=100)
    for i in range(10):
        axes[i].imshow(batch[i].mul(0.5).add(0.5).clip(0,1).permute(1,2,0))
        axes[i].set_xticks([]); axes[i].set_yticks([])
    fig.tight_layout(pad=0.1)

print('Random (unpaired) images from Colored MNIST-2 (1st row) and Colored MNIST-3 (2nd row) train sets')
plot_images(sample_mnist_2())
plot_images(sample_mnist_3())

## Architectures

In [ ]:
def build_models():
    """A fresh T and f; every training section calls this right after set_seed(SEED)."""
    T = nn.Sequential(
        nn.Conv2d(3, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.Conv2d(128, 3, kernel_size=5, padding=2),
    ).to(DEVICE)
    f = nn.Sequential(
        nn.Conv2d(3, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.AvgPool2d(2), #  128 x 8 x 8
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.AvgPool2d(2), #  256 x 4 x 4
        nn.Conv2d(128, 128, kernel_size=5, padding=2),
        nn.ReLU(True),
        nn.AvgPool2d(2), #  512 x 2 x 2
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(True),
        nn.AvgPool2d(2), #  512 x 1 x 1
        nn.Conv2d(128, 1, kernel_size=1, padding=0),
        nn.Flatten(1),
    ).to(DEVICE)
    return T, f


T, f = build_models()


print('T params:', np.sum([np.prod(p.shape) for p in T.parameters()]))
print('f params:', np.sum([np.prod(p.shape) for p in f.parameters()]))

def cost(X, T_X):
    X_flat = X.flatten(start_dim=1)
    T_X_flat = T_X.flatten(start_dim=1)

    cost = torch.pow(X_flat - T_X_flat, 2).sum(dim=1).mean()

    return cost


SEED = 0
MAX_STEPS = 50000 + 1
inner_iter = 5
# Step sizes (gamma_T, gamma_V) of each training section, chosen by a sweep: lowest
# mean per-pixel D_target over seeds (number of seeds in parentheses).
LR = {
    "eg": (2e-4, 2e-4),  # D_target 0.0609 (3 seeds)
    "tts_semi_dual": (5e-4, 1e-4),  # D_target 0.0547 (3 seeds)
    "tts_lagrangian": (2e-4, 5e-4),  # D_target 0.0587 (1 seed)
    "ogda": (2e-4, 2e-4),  # D_target 0.0551 (1 seed)
    "gdmax_semi_dual": (2e-4, 2e-4),  # D_target 0.0561 (3 seeds)
    "gdmax_lagrangian": (1e-4, 1e-4),  # D_target 0.1867 (3 seeds)
}
OGDA_BETAS = (0.0, 0.9)


## Training with `src.train`

The EG and OGDA cells below call the same training functions as notebooks 7–8 (`src/train.py`), so all experiments share one implementation of each method.

In [ ]:
import sys
sys.path.append("..")
from src.train import train_extragradient, train_gdmax, train_ogda
from src.utils import set_seed


# src.train samplers take a batch size; the loaders above already fix it to BATCH_SIZE.
def sample_mu(batch_size):
    return sample_mnist_2()


def sample_nu(batch_size):
    return sample_mnist_3()


def show_progress(step, T, f):
    with torch.no_grad():
        T.eval()
        T_X_test = T(X_test_fixed.to(DEVICE)).cpu()
        T.train()
    print('X | T(X) | Y')
    plot_images(X_test_fixed)
    plot_images(T_X_test)
    plot_images(Y_test_fixed)
    plt.show()


## EG

Extragradient with equal step sizes, $\gamma_T = \gamma_V$ (`train_extragradient(use_adam=True)`, as in notebooks 7–8): the look-ahead and the update use one batch, and the Adam state is restored before the update.

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["eg"]

history = train_extragradient(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    use_adam=True, optimizer_kwargs=dict(weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "EG")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## TTS-EG (Semi-Dual)

Two-timescale extragradient for $\max_V \min_T$: $\gamma_T > \gamma_V$.

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["tts_semi_dual"]

history = train_extragradient(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    use_adam=True, optimizer_kwargs=dict(weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "Semi-Dual_TTS-EG")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## TTS-EG (Lagrangian)

Two-timescale extragradient for $\min_T \max_V$: $\gamma_T < \gamma_V$.

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["tts_lagrangian"]

history = train_extragradient(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    use_adam=True, optimizer_kwargs=dict(weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "Lagrangian_TTS-EG")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## OGDA

Optimistic Adam (`train_ogda`, `src/optim.py`): one simultaneous gradient evaluation per step, after which `T` and `f` move by $2u_t - u_{t-1}$, where $u_t$ is the Adam step. $\beta_1 = 0$ (`OGDA_BETAS`): with momentum, optimism did not converge in the 2D experiments.

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["ogda"]

history = train_ogda(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    optimizer_kwargs=dict(betas=OGDA_BETAS, weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "OGDA")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## GDmax (Semi-Dual)

`train_gdmax`: `inner_iter` Adam descent steps on `T` per Adam ascent step on `f`, each on a fresh batch.

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["gdmax_semi_dual"]

history = train_gdmax(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f, K=inner_iter, inner="T",
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    optimizer_kwargs=dict(weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "Semi-Dual_GDmax")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## GDmax (Lagrangian)

`train_gdmax(inner="f")`: `inner_iter` Adam ascent steps on `f` per Adam descent step on `T`, each on a fresh batch. The inner maximization of the Lagrangian problem is unbounded unless $T_\sharp\mu = \nu$ (Section 5 of the paper).

In [ ]:
set_seed(SEED)
T, f = build_models()
lr_T, lr_f = LR["gdmax_lagrangian"]

history = train_gdmax(
    T, f, sample_mu, sample_nu, cost,
    n_steps=MAX_STEPS, lr_T=lr_T, lr_f=lr_f, K=inner_iter, inner="f",
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    optimizer_kwargs=dict(weight_decay=1e-10),
    log_every=500, callback=show_progress,
)

mode_name = "color_to_color"
os.makedirs(mode_name, exist_ok=True)
save_path = os.path.join(mode_name, "Lagrangian_GDmax")

T.eval()
torch.save(T.state_dict(), save_path)
print(f"✅ [{mode_name}] model is saved")


## Evaluation

Per-pixel $D_\text{target}$ and $D_\text{cost}$ with exact OT (POT) on the test split, for the `T` trained most recently.

In [ ]:
all_X_test, all_Y_test = [], []
for imgs, _ in mnist_2_test_loader:
    all_X_test.append(imgs)
for imgs, _ in mnist_3_test_loader:
    all_Y_test.append(imgs)

X_test_all = torch.cat(all_X_test, dim=0).to(DEVICE)
Y_test_all = torch.cat(all_Y_test, dim=0).to(DEVICE)


T.eval()
with torch.no_grad():
    TX_test_all = torch.clamp(T(X_test_all), -1.0, 1.0)


TX_flat = TX_test_all.view(TX_test_all.size(0), -1).cpu().numpy()
X_flat = X_test_all.view(X_test_all.size(0), -1).cpu().numpy()
Y_flat = Y_test_all.view(Y_test_all.size(0), -1).cpu().numpy()

n, m = X_flat.shape[0], Y_flat.shape[0]
a, b = np.ones((n,)) / n, np.ones((m,)) / m
pixel_dim = 3 * 16 * 16 # 768



M_tx_y = ot.dist(TX_flat, Y_flat, metric='sqeuclidean')
w2_tx_y = ot.emd2(a, b, M_tx_y)
per_pixel_w2_tx_y = w2_tx_y / pixel_dim


M_x_y = ot.dist(X_flat, Y_flat, metric='sqeuclidean')
w2_x_y = ot.emd2(a, b, M_x_y)
per_pixel_w2_x_y = w2_x_y / pixel_dim
per_pixel_transport_cost = np.mean(np.sum((X_flat - TX_flat)**2, axis=1)) / pixel_dim
per_pixel_diff = np.abs(per_pixel_w2_x_y - per_pixel_transport_cost)




print("-" * 65)
print(f"{'Metric Item (Per-Pixel)':<40} | {'Value':<15}")
print("-" * 65)
print(f"{'1. D_target':<40} | {per_pixel_w2_tx_y:.6f}")
print(f"{'2. D_cost':<40} | {per_pixel_diff:.6f}")
print("-" * 65)